# 01 — Exploratory Data Analysis

Portable EDA notebook for the fake news detection project.

**Works anywhere:** local Jupyter, VS Code, Colab, or Kaggle.  
Run the setup cell first — it finds the project root and imports from `src/`.

In [ ]:
import subprocess
import sys
from pathlib import Path

INSTALL_ON_REMOTE = False  # set True on Colab/Kaggle for first run

candidate = Path.cwd().resolve()
for path in (candidate, *candidate.parents):
    if (path / "pyproject.toml").exists():
        PROJECT_ROOT = path
        break
else:
    raise FileNotFoundError(
        "Could not find project root. Open this notebook from the repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if INSTALL_ON_REMOTE:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", ".[dev,notebook]"],
        cwd=PROJECT_ROOT,
    )

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import load_dataset_from_config
from src.data.preprocess import preprocess_text
from utils.config import load_config

sns.set_theme(style="whitegrid")

CONFIG_PATH = PROJECT_ROOT / "configs/baseline.yaml"
config = load_config(CONFIG_PATH)

texts, labels = load_dataset_from_config(config["dataset"], PROJECT_ROOT)

df = pd.DataFrame({"text": texts, "label": labels})
df["label_name"] = df["label"].map({0: "real", 1: "fake"})
df["char_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

print(f"Loaded {len(df):,} articles from {config['dataset']['path']}")
df.head()

## Dataset overview

In [ ]:
summary = {
    "rows": len(df),
    "real_count": int((df["label"] == 0).sum()),
    "fake_count": int((df["label"] == 1).sum()),
    "avg_char_length": round(df["char_length"].mean(), 2),
    "avg_word_count": round(df["word_count"].mean(), 2),
}
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="label_name", ax=axes[0])
axes[0].set_title("Class distribution")
axes[0].set_xlabel("Label")

sns.histplot(data=df, x="word_count", hue="label_name", kde=True, ax=axes[1])
axes[1].set_title("Word count distribution")
axes[1].set_xlabel("Words per article")

plt.tight_layout()
plt.show()

## Sample headlines

In [ ]:
for label_name in ["real", "fake"]:
    print(f"\n=== {label_name.upper()} samples ===")
    samples = df.loc[df["label_name"] == label_name, "text"].head(3)
    for idx, text in enumerate(samples, start=1):
        cleaned = preprocess_text(text, config.get("preprocessing"))
        print(f"{idx}. {text}")
        print(f"   cleaned: {cleaned}")

## Next step

Open `02_baseline_models.ipynb` to train and compare classical models on this dataset.